# 3D CNN (ResNet-18) Trainer
- Use this notebook file to directly train from the video dataset without conversion to images
- Trains based on ResNet-18 3D architecture CNN (This allows direct video data training)

In [45]:
#Cell 1

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights
from decord import VideoReader, cpu
import numpy as np
import os
import random
from collections import defaultdict
from sklearn.model_selection import train_test_split
import torch.nn.functional as F
from sklearn.utils import resample
import matplotlib.pyplot as plt
from datetime import datetime

In [46]:
#Cell 2

# Specify dataset root with absolute path (adjust to your exact location)
DATA_ROOT = '/home/heytanix/PCL_repository/violence-detection-dataset/'  # Example; change if different

In [47]:
#Cell 3

# Class-specific directories (each contains cam1/ and cam2/ with MP4s)
VIOLENT_DIR = os.path.join(DATA_ROOT, 'violent')
NON_VIOLENT_DIR = os.path.join(DATA_ROOT, 'non-violent')

In [48]:
#Cell 4

# Quick check for directory existence
print(f"Current notebook working directory: {os.getcwd()}")
for dir_path in [DATA_ROOT, NON_VIOLENT_DIR, VIOLENT_DIR]:
    if os.path.exists(dir_path):
        print(f"{dir_path} exists.")
    else:
        print(f"WARNING: {dir_path} does NOT exist. Check your path!")

Current notebook working directory: /home/heytanix/PCL_repository
/home/heytanix/PCL_repository/violence-detection-dataset/ exists.
/home/heytanix/PCL_repository/violence-detection-dataset/non-violent exists.
/home/heytanix/PCL_repository/violence-detection-dataset/violent exists.


In [49]:
#Cell 5

# CSV files (for reference; not used in binary classification but can be loaded if needed)
VIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'violent-action-classes.csv')
NONVIOLENT_CLASSES_CSV = os.path.join(DATA_ROOT, 'nonviolent-action-classes.csv')
ACTION_OCCURRENCES_CSV = os.path.join(DATA_ROOT, 'action-class-occurrences.csv')

In [50]:
#Cell 6

# Class labels
CLASSES = ['non-violent', 'violent']  # Match your folder names

In [51]:
#Cell 7

def find_mp4_files(directory):
    mp4_files = []
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist. Skipping.")
        return mp4_files
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith('.mp4'):
                full_path = os.path.join(root, file)
                mp4_files.append(full_path)
    return mp4_files

non_violent_mp4s = find_mp4_files(NON_VIOLENT_DIR)
violent_mp4s = find_mp4_files(VIOLENT_DIR)

print(f"Found {len(non_violent_mp4s)} non-violent MP4s")
print(f"Found {len(violent_mp4s)} violent MP4s")

if not non_violent_mp4s and not violent_mp4s:
    raise ValueError("No MP4 files found! Check your DATA_ROOT path.")

# Group by scene ID to prevent cam1/cam2 data leakage
# e.g., cam1/5.mp4 and cam2/5.mp4 are the SAME scene — must stay in same split
scene_to_paths = defaultdict(list)
for path in non_violent_mp4s:
    scene_id = os.path.basename(path)  # e.g., "5.mp4"
    scene_to_paths[('non-violent', scene_id)].append((path, 0))

for path in violent_mp4s:
    scene_id = os.path.basename(path)
    scene_to_paths[('violent', scene_id)].append((path, 1))

# Split at scene level
scene_keys = list(scene_to_paths.keys())
scene_labels = [k[0] for k in scene_keys]  # for stratification

train_val_keys, test_keys = train_test_split(
    scene_keys, test_size=0.15, stratify=scene_labels, random_state=42
)
train_val_labels = [k[0] for k in train_val_keys]
train_keys, val_keys = train_test_split(
    train_val_keys, test_size=0.176, stratify=train_val_labels, random_state=42
)

# Expand scenes back to individual file paths
def expand_keys(keys, scene_map):
    data = []
    for k in keys:
        data.extend(scene_map[k])
    return data

train_data = expand_keys(train_keys, scene_to_paths)
val_data   = expand_keys(val_keys,   scene_to_paths)
test_data  = expand_keys(test_keys,  scene_to_paths)

print(f"Train samples: {len(train_data)}, Val samples: {len(val_data)}, Test samples: {len(test_data)}")


Found 120 non-violent MP4s
Found 230 violent MP4s
Train samples: 242, Val samples: 54, Test samples: 54


In [52]:
#Cell 8

# Separate by class
train_non_violent = [(p, l) for p, l in train_data if l == 0]
train_violent     = [(p, l) for p, l in train_data if l == 1]

print(f"Before balancing — Non-violent: {len(train_non_violent)}, Violent: {len(train_violent)}")

# Oversample the minority class (non-violent) instead of discarding violent data
max_count = max(len(train_non_violent), len(train_violent))

if len(train_non_violent) < max_count:
    train_non_violent = resample(train_non_violent, n_samples=max_count, replace=True, random_state=42)
elif len(train_violent) < max_count:
    train_violent = resample(train_violent, n_samples=max_count, replace=True, random_state=42)

train_data_balanced = train_non_violent + train_violent
random.shuffle(train_data_balanced)

print(f"After balancing — Total train samples: {len(train_data_balanced)}")
print(f"Non-violent: {sum(1 for _, l in train_data_balanced if l == 0)}, Violent: {sum(1 for _, l in train_data_balanced if l == 1)}")

Before balancing — Non-violent: 84, Violent: 158
After balancing — Total train samples: 316
Non-violent: 158, Violent: 158


In [53]:
#Cell 9

class VideoDataset(Dataset):
    def __init__(self, data, clip_len=16, transform=None):
        self.video_paths = [d[0] for d in data]
        self.labels      = [d[1] for d in data]
        self.clip_len    = clip_len
        self.transform   = transform

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        try:
            vr = VideoReader(self.video_paths[idx], ctx=cpu(0))
            total_frames = len(vr)

            if total_frames >= self.clip_len:
                indices = np.linspace(0, total_frames - 1, self.clip_len, dtype=int)
            else:
                indices = list(range(total_frames))
                indices += [total_frames - 1] * (self.clip_len - total_frames)

            frames = vr.get_batch(indices).asnumpy().copy()  # .copy() forces contiguous memory
            frames = torch.from_numpy(frames).float() / 255.0
            frames = frames.permute(3, 0, 1, 2).contiguous()  # (C, T, H, W), contiguous

            if self.transform:
                frames = self.transform(frames)

            return frames, self.labels[idx]

        except Exception as e:
            print(f"Warning: Failed to load {self.video_paths[idx]}: {e}")
            return torch.zeros(3, self.clip_len, 112, 112), self.labels[idx]

In [54]:
#Cell 10

class ResizeVideo:
    def __init__(self, size):
        self.size = size  # (H, W)

    def __call__(self, video):  # (C, T, H, W)
        C, T, H, W = video.shape
        video = video.view(C * T, H, W).unsqueeze(0)
        video = F.interpolate(video, size=self.size, mode='bilinear', align_corners=False)
        video = video.squeeze(0).view(C, T, *self.size)
        return video


class NormalizeVideo:
    def __init__(self, mean, std):
        self.mean = torch.tensor(mean).view(3, 1, 1, 1)
        self.std  = torch.tensor(std).view(3, 1, 1, 1)

    def __call__(self, video):  # (C, T, H, W)
        return (video - self.mean) / self.std


class VideoAugmentation:
    def __call__(self, video):  # (C, T, H, W)
        # Random horizontal flip
        if random.random() > 0.5:
            video = torch.flip(video, dims=[3])

        # Brightness + contrast jitter
        brightness = random.uniform(0.8, 1.2)
        contrast   = random.uniform(0.8, 1.2)
        video = torch.clamp(video * brightness, 0, 1)
        mean  = video.mean(dim=[2, 3], keepdim=True)
        video = torch.clamp((video - mean) * contrast + mean, 0, 1)

        # Random Gaussian noise
        if random.random() > 0.5:
            video = torch.clamp(video + torch.randn_like(video) * 0.02, 0, 1)

        # Random temporal jitter — drop a few frames from start or end
        T = video.shape[1]
        max_drop = max(1, T // 8)
        drop = random.randint(0, max_drop)
        if random.random() > 0.5:
            video = video[:, drop:, :, :]
        else:
            video = video[:, :T - drop, :, :]

        # Re-interpolate back to original T length
        C, t, H, W = video.shape
        if t != T:
            video = video.unsqueeze(0)   # (1, C, t, H, W)
            video = F.interpolate(video, size=(T, H, W), mode='trilinear', align_corners=False)
            video = video.squeeze(0)     # (C, T, H, W)

        return video


class VideoTransformCompose:
    def __init__(self, transforms_list):
        self.transforms_list = transforms_list

    def __call__(self, video):
        for t in self.transforms_list:
            video = t(video)
        return video


# ImageNet mean/std (used by R3D-18 pretrained on Kinetics)
MEAN = [0.43216, 0.394666, 0.37645]
STD  = [0.22803, 0.22145,  0.216989]

transform_train = VideoTransformCompose([
    ResizeVideo((128, 128)),
    VideoAugmentation(),
    ResizeVideo((112, 112)),
    NormalizeVideo(MEAN, STD),
])

transform_val_test = VideoTransformCompose([
    ResizeVideo((112, 112)),
    NormalizeVideo(MEAN, STD),
])

# Hyperparameters
CLIP_LEN    = 16
BATCH_SIZE  = 8
NUM_EPOCHS  = 20
NUM_WORKERS = 4

train_dataset = VideoDataset(train_data_balanced, clip_len=CLIP_LEN, transform=transform_train)
val_dataset   = VideoDataset(val_data,            clip_len=CLIP_LEN, transform=transform_val_test)
test_dataset  = VideoDataset(test_data,           clip_len=CLIP_LEN, transform=transform_val_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}, Test batches: {len(test_loader)}")

Training batches: 40, Validation batches: 7, Test batches: 7


In [55]:
#Cell 11

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load pretrained R3D-18 (non-deprecated API)
model = r3d_18(weights=R3D_18_Weights.KINETICS400_V1)

# Replace classifier head with Dropout + Linear
model.fc = nn.Sequential(
    nn.Dropout(p=0.5),
    nn.Linear(model.fc.in_features, 2)
)

# Freeze backbone — only train the classifier head initially
for name, param in model.named_parameters():
    if 'fc' not in name:
        param.requires_grad = False

model = model.to(device)

# Differential learning rates + weight decay
backbone_params   = [p for n, p in model.named_parameters() if 'fc' not in n]
classifier_params = [p for n, p in model.named_parameters() if 'fc'     in n]

optimizer = optim.Adam([
    {'params': backbone_params,   'lr': 1e-4, 'weight_decay': 1e-4},
    {'params': classifier_params, 'lr': 1e-3, 'weight_decay': 1e-4},
])

# ReduceLROnPlateau watches val_loss and reduces LR when it stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)

criterion = nn.CrossEntropyLoss()

print(f"Model ready. Backbone frozen — training classifier head only for first 3 epochs.")
print(f"Total trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Using device: cuda
Model ready. Backbone frozen — training classifier head only for first 3 epochs.
Total trainable params: 1,026


In [56]:
#Cell 12

UNFREEZE_EPOCH = 3
PATIENCE       = 5

best_val_acc      = 0.0
best_val_loss     = float('inf')
epochs_no_improve = 0
best_epoch        = 1

train_losses, val_losses, val_accuracies = [], [], []

print(f"Starting training for {NUM_EPOCHS} epochs...")
print(f"Training batches: {len(train_loader)}, Validation batches: {len(val_loader)}")

for epoch in range(NUM_EPOCHS):
    epoch_start = datetime.now()

    if epoch == UNFREEZE_EPOCH:
        for param in model.parameters():
            param.requires_grad = True
        print(f"\nEpoch {epoch+1}: Backbone unfrozen. Full model now training.")

    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for batch_idx, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * videos.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)

        if (batch_idx + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Batch [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

    avg_train_loss = running_loss / total
    train_acc      = 100.0 * correct / total

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)
            outputs = model(videos)
            loss    = criterion(outputs, labels)
            val_loss    += loss.item() * videos.size(0)
            _, predicted = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    avg_val_loss = val_loss / val_total
    val_acc      = 100.0 * val_correct / val_total

    scheduler.step(avg_val_loss)

    elapsed    = (datetime.now() - epoch_start).seconds
    current_lr = optimizer.param_groups[-1]['lr']

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}] ({elapsed}s):")
    print(f"  Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {avg_val_loss:.4f}, Val Acc: {val_acc:.2f}%")
    print(f"  LR: {current_lr:.6f}")
    print("-" * 60)

    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = avg_val_loss
        best_epoch    = epoch + 1
        torch.save(model.state_dict(), 'best_model_weights.pth')
        print(f"  ✓ New best val accuracy: {best_val_acc:.2f}% — model saved.")

    if avg_val_loss < best_val_loss and epoch >= UNFREEZE_EPOCH:
        epochs_no_improve = 0
    elif epoch >= UNFREEZE_EPOCH:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}.")
            break

print(f"\nTraining complete! Best val accuracy: {best_val_acc:.2f}% at epoch {best_epoch}")
model.load_state_dict(torch.load('best_model_weights.pth'))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(train_losses, label='Train Loss')
ax1.plot(val_losses,   label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(val_accuracies, label='Val Accuracy')
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend()
plt.tight_layout()
plt.show()


Starting training for 20 epochs...
Training batches: 40, Validation batches: 7


KeyboardInterrupt: 

In [ ]:
#Cell 13

torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'classes': CLASSES,
    'clip_len': CLIP_LEN,
    'input_size': 112,
    'mean': MEAN,
    'std': STD,
    'best_epoch': best_epoch,
    'best_val_accuracy': best_val_acc,
}, 'violence_classifier.pth')

print(f"Model saved as 'violence_classifier.pth'")
print(f"Best validation accuracy: {best_val_acc:.2f}% at epoch {best_epoch}")